# Leakage-Free CIPN Prediction Pipeline

Per-fold band and feature selection — nothing from the outer test set ever leaks
into scaler means, feature rankings, hyperparameter choices, or thresholds.

## Design (aligned with methodology discussion, 2026-04-20)

1. **Outcome** — binary: `Diff = T2 - T1`, class 0 = improver (`Diff < -2`), class 1 = non-improver.
2. **Candidate feature pool** — for every patient, for every (sheet, band, statistic),
   compute one scalar. Statistics: `mean`, `std`, `median`, `p90` across channels/pairs.
   Nine sheets × their bands × 4 stats → a ~300-dim candidate pool per patient.
3. **Clinical block** (always kept, never filtered): T1 pain, age, sex, group, neuropathy months.
4. **Outer CV** — 5-fold `StratifiedGroupKFold` keyed on patient id.
5. **Inside each outer fold**:
   1. Run `INNER_N_SPLITS`-fold inner CV *only* on outer-train patients.
   2. For each inner fold, score every candidate feature with a univariate LR
      (+ StandardScaler fit on the inner-train). Aggregate mean balanced
      accuracy across inner folds.
   3. Keep candidates whose inner mean BA ≥ `ACC_FLOOR` **then** take the top
      `K = n_train // FEATURES_PER_SAMPLE` by score.
   4. Concatenate selected EEG features with clinical features; grid-search an
      SVM-RBF / LogReg-L2 / XGB head with `StratifiedGroupKFold` on outer-train only.
   5. Calibrate threshold on outer-train predictions; evaluate on outer-test.
   6. Compute SHAP on the outer-test predictions; save per-fold `.npz`.
6. **Leakage guards** — every scaler, imputer, PCA, and feature rank is fit only
   on outer-train inner-train data. All z-scored sheets (`Z_*`) are avoided in
   the EEG pool because their z-scores were computed across the full cohort.
7. **Outputs** — `reports/leakage_free_v3/`:
   - `fold_results.csv`, `per_fold_selected_features.csv`
   - `fold{k}_shap.npz`, `fold{k}_shap_beeswarm.png`
   - `oof_confusion.png`, `oof_roc.png`, `aggregated_shap.png`

In [ ]:
import os, re, glob, warnings, json
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.metrics import (
    balanced_accuracy_score, roc_auc_score, f1_score,
    recall_score, confusion_matrix, roc_curve,
)

import shap
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
print('imports OK — sklearn / xgboost / shap ready')

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR = Path('processeddata')
OUT_DIR  = Path('reports/leakage_free_v3')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Outcome definition ───────────────────────────────────────────────────────
THRESHOLD = -2.0     # Diff < -2 → improver (class 0)

# ── CV ───────────────────────────────────────────────────────────────────────
OUTER_N_SPLITS = 5   # methodology: 5-fold outer CV
INNER_N_SPLITS = 5   # used for both feature scoring and hyperparameter search
RANDOM_STATE   = 42

# ── Feature-selection knobs ──────────────────────────────────────────────────
# methodology: max features ≈ n_train / 10
FEATURES_PER_SAMPLE = 10
ACC_FLOOR           = 0.55  # discard candidates below this inner-mean BA
TOP_K_CAP           = 12    # hard upper bound regardless of n_train

STATISTICS = ('mean', 'std', 'median', 'p90')

# Only use RAW sheets — Z_* sheets were z-scored across the full cohort
# so they carry test-fold information. StandardScaler inside the pipe
# (fit only on outer-train) replaces that z-scoring cleanly.
BANDS_14 = ['Delta', 'Theta', 'Alpha', 'Beta', 'HighBeta', 'Gamma',
            'HighGamma', 'Alpha1', 'Alpha2', 'Beta1', 'Beta2', 'Beta3',
            'Gamma1', 'Gamma2']
BANDS_10 = ['Delta', 'Theta', 'Alpha', 'Beta', 'HighBeta',
            'Alpha1', 'Alpha2', 'Beta1', 'Beta2', 'Beta3']

# 5-Hz buckets covering 1–50 Hz — used to summarise the per-Hz sheets
HZ_BUCKETS = [(lo, lo + 5) for lo in range(1, 50, 5)]   # (1,6), (6,11), ..., (46,51)

SHEET_CONFIGS = {
    # ── band-based sheets (19 channels × bands  or  171 pairs × bands) ────────
    'FFT_abs_bandpower_uV2': {'type': 'band',   'bands':   BANDS_14},
    'FFT_rel_bandpower_pct': {'type': 'band',   'bands':   BANDS_14},
    'PeakFreq_Hz':           {'type': 'band',   'bands':   BANDS_14},
    'FFT_Coherence':         {'type': 'band',   'bands':   BANDS_10},
    'FFT_PhaseLag_PLI':      {'type': 'band',   'bands':   BANDS_10},
    # ── per-Hz sheets (19 channels × 50 Hz bins) — bucket into 5-Hz ranges ───
    'FFT_abs_1to50Hz_uV2':   {'type': 'per_hz', 'buckets': HZ_BUCKETS},
    'FFT_rel_1to50Hz_pct':   {'type': 'per_hz', 'buckets': HZ_BUCKETS},
}

# Z_* sheets intentionally omitted — their values were z-scored across the
# entire cohort (test-fold patients included). The equivalent leakage-free
# transform is StandardScaler fit inside each Pipeline on outer-train only,
# which is already part of every head below.

CLINICAL_COLS = [
    'T1_pain', 'age', 'sex_male', 'group_NFB', 'group_NFB_DL',
    'group_DL', 'neuropathy_months',
]

n_band = sum(len(cfg['bands'])    for cfg in SHEET_CONFIGS.values() if cfg['type'] == 'band')
n_hz   = sum(len(cfg['buckets'])  for cfg in SHEET_CONFIGS.values() if cfg['type'] == 'per_hz')
print('Config:')
print(f'  Outer folds            : {OUTER_N_SPLITS}')
print(f'  Inner folds            : {INNER_N_SPLITS}')
print(f'  Statistics per slot    : {STATISTICS}')
print(f'  Sheets ({len(SHEET_CONFIGS)})       :')
for s, cfg in SHEET_CONFIGS.items():
    tag = f"{len(cfg['bands'])} bands" if cfg['type']=='band' else f"{len(cfg['buckets'])} Hz buckets"
    print(f'     {s:<28s} {cfg["type"]:<7s} {tag}')
print(f'  Total candidate slots  : {(n_band + n_hz) * len(STATISTICS)}')
print(f'  Features per sample    : 1 / {FEATURES_PER_SAMPLE} (≥ {ACC_FLOOR} BA)')
print(f'  Hard cap on selected K : {TOP_K_CAP}')

In [ ]:
# ── Outcomes + clinical features ─────────────────────────────────────────────

raw = pd.read_excel(DATA_DIR / 'Randomization factors and Primary outcome.xlsx')

# Diff = T2 - T1 pain unpleasantness
pain = (
    raw[raw['Event Name'].isin(['T1', 'T2'])]
    .pivot_table(index='Patient number', columns='Event Name',
                 values='Pain Unpleasantness')
    .dropna()
)
pain['Diff'] = pain['T2'] - pain['T1']

# Clinical block from T1 rows
t1 = raw[raw['Event Name'] == 'T1'].copy()
group_col = 'Group assignment - can code the variables (NFB=1; NFB+DL=2; DL=3)'
neuro_col = 'How many months have you been experiencing neuropathy?'

clinical = pd.DataFrame(index=t1['Patient number'].values)
clinical['T1_pain']           = t1['Pain Unpleasantness'].values
clinical['age']               = pd.to_numeric(t1['Age'], errors='coerce').values
clinical['sex_male']          = (t1['Sex:'].astype(str).str.strip().str.lower() == 'male').astype(int).values
grp = t1[group_col].astype(str).str.strip().str.upper()
clinical['group_NFB']         = (grp == 'NFB').astype(int).values
clinical['group_NFB_DL']      = (grp == 'NFB+DL').astype(int).values
clinical['group_DL']          = (grp == 'DL').astype(int).values
clinical['neuropathy_months'] = pd.to_numeric(t1[neuro_col], errors='coerce').values

# keep only rows with a valid Diff (outcome present)
clinical = clinical.loc[clinical.index.intersection(pain.index)].copy()
clinical = clinical[~clinical.index.duplicated(keep='first')]
clinical = clinical[CLINICAL_COLS]

# EEG file index
eeg_files = {}
for f in sorted(DATA_DIR.glob('CIPN3*.xlsx')):
    m = re.search(r'CIPN3(\d+)', f.name)
    if m:
        pid = int(m.group(1))
        eeg_files.setdefault(pid, f)

# Final cohort: has EEG file AND valid outcome AND clinical row
cohort_ids = sorted(set(eeg_files) & set(pain.index) & set(clinical.index))
y_cont     = pain.loc[cohort_ids, 'Diff'].values
y          = (y_cont >= THRESHOLD).astype(int)           # 1 = non-improver
groups     = np.array(cohort_ids, dtype=int)

print(f'Cohort     : {len(cohort_ids)} patients')
print(f'Class 0    : {int((y == 0).sum())} (improvers, Diff < {THRESHOLD})')
print(f'Class 1    : {int((y == 1).sum())} (non-improvers, Diff >= {THRESHOLD})')
print(f'Diff mean  : {y_cont.mean():.2f} | SD={y_cont.std():.2f}')

In [ ]:
# ── Build candidate feature pool ─────────────────────────────────────────────
# For each sheet, emit one scalar per (band or Hz-bucket, stat) per patient.
# Column name convention: <sheet>__<slot>__<stat>
#   band sheets   →  slot = band name   (e.g. "Alpha1")
#   per-Hz sheets →  slot = "Hz{lo}-{hi}"  (e.g. "Hz1-6")

def summarize(arr, stat):
    arr = arr[np.isfinite(arr)]
    if len(arr) == 0:
        return np.nan
    if stat == 'mean':   return float(arr.mean())
    if stat == 'std':    return float(arr.std())
    if stat == 'median': return float(np.median(arr))
    if stat == 'p90':    return float(np.percentile(arr, 90))
    raise ValueError(stat)


def _slot_array_band(df_s, band):
    """Return the flat array of values for a single band in a band sheet."""
    if band not in df_s.columns:
        return np.array([])
    return df_s[band].values.astype(float)


def _slot_array_per_hz(df_s, lo, hi):
    """
    Return the flat array of values in the Hz columns in [lo, hi).
    Accepts column names like '1Hz', '2Hz', ... — ignores missing bins.
    """
    keep_cols = []
    for c in df_s.columns:
        m = re.match(r'^(\d+)Hz$', str(c))
        if m and lo <= int(m.group(1)) < hi:
            keep_cols.append(c)
    if not keep_cols:
        return np.array([])
    return df_s[keep_cols].values.astype(float).ravel()


def patient_feature_row(xlsx_path, sheet_configs, statistics):
    """Return one row (dict) of candidate features for a single patient."""
    row = {}
    for sheet, cfg in sheet_configs.items():
        try:
            df_s = pd.read_excel(xlsx_path, sheet_name=sheet, index_col=0)
        except Exception:
            # emit NaNs for every slot so column set is consistent
            if cfg['type'] == 'band':
                slots = [(b, b) for b in cfg['bands']]
            else:
                slots = [(f'Hz{lo}-{hi}', (lo, hi)) for (lo, hi) in cfg['buckets']]
            for slot_name, _ in slots:
                for st in statistics:
                    row[f'{sheet}__{slot_name}__{st}'] = np.nan
            continue

        if cfg['type'] == 'band':
            for b in cfg['bands']:
                arr = _slot_array_band(df_s, b)
                for st in statistics:
                    row[f'{sheet}__{b}__{st}'] = summarize(arr, st) if len(arr) else np.nan
        else:  # per_hz
            for lo, hi in cfg['buckets']:
                slot_name = f'Hz{lo}-{hi}'
                arr = _slot_array_per_hz(df_s, lo, hi)
                for st in statistics:
                    row[f'{sheet}__{slot_name}__{st}'] = summarize(arr, st) if len(arr) else np.nan
    return row


print('Building per-patient feature pool (this reads every xlsx once)...')
pool_rows = []
for pid in cohort_ids:
    r = patient_feature_row(eeg_files[pid], SHEET_CONFIGS, STATISTICS)
    r['Patient number'] = pid
    pool_rows.append(r)

X_pool = pd.DataFrame(pool_rows).set_index('Patient number')
X_pool = X_pool.loc[cohort_ids]   # guarantee row order matches y/groups

# Align clinical features to the same patient order
X_clin = clinical.loc[cohort_ids].copy()

# Audit: slot counts per sheet
print(f'\nFeature pool shape : {X_pool.shape}  (rows=patients, cols=EEG candidates)')
print(f'Clinical block     : {X_clin.shape}')
print(f'NaN rate in pool   : {X_pool.isna().mean().mean():.3%}\n')
print('Per-sheet slot counts:')
for sheet in SHEET_CONFIGS:
    n = sum(c.startswith(sheet + '__') for c in X_pool.columns)
    print(f'  {sheet:<28s}  {n:>3d} candidate features')

In [ ]:
# ── Per-fold feature selection ───────────────────────────────────────────────
# Univariate LR scored with StratifiedGroupKFold on outer-train only.

def score_candidate_univariate(x_col, y_in, groups_in, inner_cv):
    """
    Return mean balanced accuracy of a single-feature LogReg classifier
    across inner CV folds. Impute+scale are fit only on each inner train fold.
    """
    x = x_col.reshape(-1, 1)
    scores = []
    for tr_i, va_i in inner_cv.split(x, y_in, groups=groups_in):
        pipe = Pipeline([
            ('imp', SimpleImputer(strategy='median')),
            ('scl', StandardScaler()),
            ('clf', LogisticRegression(
                class_weight='balanced', max_iter=5000,
                solver='liblinear', random_state=RANDOM_STATE)),
        ])
        try:
            pipe.fit(x[tr_i], y_in[tr_i])
            scores.append(
                balanced_accuracy_score(y_in[va_i], pipe.predict(x[va_i])))
        except Exception:
            scores.append(np.nan)
    return float(np.nanmean(scores)) if scores else np.nan


def select_features_in_fold(X_tr_pool, y_tr, groups_tr,
                            inner_cv, acc_floor, top_k):
    """
    Score every EEG candidate univariately on outer-train only.
    Keep those with inner-mean BA >= acc_floor, take top-k by score.
    Returns the list of column names chosen and their scores.
    """
    scores = {}
    for col in X_tr_pool.columns:
        vals = X_tr_pool[col].values
        if np.all(~np.isfinite(vals)):
            continue
        s = score_candidate_univariate(vals, y_tr, groups_tr, inner_cv)
        if np.isfinite(s):
            scores[col] = s

    ranked   = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)
    survivors = [(c, s) for c, s in ranked if s >= acc_floor]
    # Fall back: if nothing passes the floor, take the top-k regardless
    chosen    = survivors[:top_k] if survivors else ranked[:top_k]
    return [c for c, _ in chosen], dict(chosen)


print('Selection helpers defined.')

In [ ]:
# ── Model heads and hyperparameter grids ─────────────────────────────────────
# Everything is wrapped in a sklearn Pipeline so imputer + scaler are fit only
# on each inner-train fold during GridSearchCV.

def make_svm():
    return (
        Pipeline([
            ('imp', SimpleImputer(strategy='median')),
            ('scl', StandardScaler()),
            ('clf', SVC(kernel='rbf', class_weight='balanced',
                        probability=True, random_state=RANDOM_STATE)),
        ]),
        {'clf__C':     [0.1, 1, 3, 10],
         'clf__gamma': ['scale', 'auto', 0.01, 0.1]},
    )


def make_logreg():
    return (
        Pipeline([
            ('imp', SimpleImputer(strategy='median')),
            ('scl', StandardScaler()),
            ('clf', LogisticRegression(
                class_weight='balanced', solver='liblinear',
                max_iter=10000, random_state=RANDOM_STATE)),
        ]),
        {'clf__penalty': ['l1', 'l2'],
         'clf__C':       [0.01, 0.1, 1, 10, 100]},
    )


def make_xgb(scale_pos_weight):
    return (
        Pipeline([
            ('imp', SimpleImputer(strategy='median')),
            ('clf', XGBClassifier(
                tree_method='hist', eval_metric='logloss',
                scale_pos_weight=scale_pos_weight,
                random_state=RANDOM_STATE, n_jobs=1)),
        ]),
        {'clf__n_estimators': [200, 400],
         'clf__max_depth':    [3, 5],
         'clf__learning_rate': [0.05, 0.1],
         'clf__subsample':    [0.8, 1.0]},
    )


MODEL_NAMES = ('SVM', 'LogReg', 'XGB')


def build_heads(y_train):
    neg = int((y_train == 0).sum())
    pos = int((y_train == 1).sum())
    spw = neg / max(pos, 1)
    return {
        'SVM':    make_svm(),
        'LogReg': make_logreg(),
        'XGB':    make_xgb(scale_pos_weight=spw),
    }


print('Model head factory defined.')

In [ ]:
# ── Main leakage-free CV loop ────────────────────────────────────────────────

outer_cv = StratifiedGroupKFold(
    n_splits=OUTER_N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

fold_records          = []   # tidy per-fold, per-model metrics
per_fold_selections   = []   # audit trail of which features were chosen
per_fold_shap         = {}   # fold -> dict(model -> (shap_values, X_test, feature_names))
oof_pred              = {m: np.full(len(y), np.nan) for m in MODEL_NAMES}
oof_prob              = {m: np.full(len(y), np.nan) for m in MODEL_NAMES}
oof_ensemble_prob     = np.full(len(y), np.nan)

for fold_i, (tr_idx, te_idx) in enumerate(
        outer_cv.split(np.zeros((len(y), 1)), y, groups=groups), start=1):

    print('\n' + '=' * 88)
    print(f'[Outer fold {fold_i}/{OUTER_N_SPLITS}]  '
          f'train={len(tr_idx)}  test={len(te_idx)}  '
          f'(cls0_tr={int((y[tr_idx]==0).sum())}, cls1_tr={int((y[tr_idx]==1).sum())})')
    assert set(groups[tr_idx]).isdisjoint(set(groups[te_idx])), 'patient leakage!'

    X_tr_pool = X_pool.iloc[tr_idx]
    X_te_pool = X_pool.iloc[te_idx]
    X_tr_clin = X_clin.iloc[tr_idx]
    X_te_clin = X_clin.iloc[te_idx]
    y_tr, y_te = y[tr_idx], y[te_idx]
    g_tr       = groups[tr_idx]

    # ── Stage 1 — per-fold feature selection on outer-train only ─────────────
    inner_cv_sel = StratifiedGroupKFold(
        n_splits=INNER_N_SPLITS, shuffle=True, random_state=RANDOM_STATE + fold_i)
    top_k = min(TOP_K_CAP, max(5, len(tr_idx) // FEATURES_PER_SAMPLE))
    selected, sel_scores = select_features_in_fold(
        X_tr_pool, y_tr, g_tr, inner_cv_sel, ACC_FLOOR, top_k)

    print(f'  Selected {len(selected)}/{X_pool.shape[1]} EEG features '
          f'(cap={top_k}, floor={ACC_FLOOR})')
    for name, sc in sorted(sel_scores.items(), key=lambda kv: -kv[1]):
        print(f'    {name:<55s}  inner BA = {sc:.3f}')

    # Build the final training/test matrices: clinical + selected EEG
    X_tr_full = pd.concat(
        [X_tr_clin.reset_index(drop=True),
         X_tr_pool[selected].reset_index(drop=True)], axis=1)
    X_te_full = pd.concat(
        [X_te_clin.reset_index(drop=True),
         X_te_pool[selected].reset_index(drop=True)], axis=1)
    feature_names = list(X_tr_full.columns)

    # ── Stage 2 — hyperparameter search per head on outer-train only ─────────
    inner_cv_grid = StratifiedGroupKFold(
        n_splits=INNER_N_SPLITS, shuffle=True,
        random_state=RANDOM_STATE + 100 + fold_i)

    heads = build_heads(y_tr)
    fitted = {}
    per_model_prob_tr = {}
    per_model_prob_te = {}

    for m_name, (pipe, grid) in heads.items():
        gs = GridSearchCV(
            pipe, grid, scoring='balanced_accuracy',
            cv=inner_cv_grid, n_jobs=-1, refit=True, error_score=np.nan)
        gs.fit(X_tr_full.values, y_tr, groups=g_tr)
        fitted[m_name] = gs.best_estimator_
        per_model_prob_tr[m_name] = gs.best_estimator_.predict_proba(X_tr_full.values)[:, 1]
        per_model_prob_te[m_name] = gs.best_estimator_.predict_proba(X_te_full.values)[:, 1]
        print(f'  [{m_name}]  inner BA={gs.best_score_:.3f}  best={gs.best_params_}')

    # ── Stage 3 — threshold calibration + test evaluation per head ───────────
    for m_name in MODEL_NAMES:
        p_tr = per_model_prob_tr[m_name]
        p_te = per_model_prob_te[m_name]
        # tune threshold on outer-train only
        ths = np.linspace(0.1, 0.9, 81)
        ba_tr = [balanced_accuracy_score(y_tr, (p_tr >= t).astype(int)) for t in ths]
        thr = float(ths[int(np.argmax(ba_tr))])

        yhat_te = (p_te >= thr).astype(int)
        oof_pred[m_name][te_idx] = yhat_te
        oof_prob[m_name][te_idx] = p_te

        rec = {
            'fold':         fold_i,
            'model':        m_name,
            'n_train':      len(tr_idx),
            'n_test':       len(te_idx),
            'n_features':   len(feature_names),
            'threshold':    thr,
            'ba':           balanced_accuracy_score(y_te, yhat_te),
            'auc':          roc_auc_score(y_te, p_te) if len(set(y_te)) > 1 else np.nan,
            'f1':           f1_score(y_te, yhat_te, zero_division=0),
            'sensitivity':  recall_score(y_te, yhat_te, pos_label=1, zero_division=0),
            'specificity':  recall_score(y_te, yhat_te, pos_label=0, zero_division=0),
        }
        print(f'  [{m_name}] TEST  BA={rec["ba"]:.3f}  AUC={rec["auc"]:.3f}  '
              f'F1={rec["f1"]:.3f}  Sens={rec["sensitivity"]:.3f}  '
              f'Spec={rec["specificity"]:.3f}  thr={thr:.2f}')
        fold_records.append(rec)

    # Soft-vote ensemble probabilities (average of three head probabilities)
    ens_te = np.mean([per_model_prob_te[m] for m in MODEL_NAMES], axis=0)
    ens_tr = np.mean([per_model_prob_tr[m] for m in MODEL_NAMES], axis=0)
    ths = np.linspace(0.1, 0.9, 81)
    ba_tr = [balanced_accuracy_score(y_tr, (ens_tr >= t).astype(int)) for t in ths]
    thr_ens = float(ths[int(np.argmax(ba_tr))])
    oof_ensemble_prob[te_idx] = ens_te
    yhat_ens = (ens_te >= thr_ens).astype(int)
    fold_records.append({
        'fold':        fold_i,
        'model':       'ENS',
        'n_train':     len(tr_idx),
        'n_test':      len(te_idx),
        'n_features':  len(feature_names),
        'threshold':   thr_ens,
        'ba':          balanced_accuracy_score(y_te, yhat_ens),
        'auc':         roc_auc_score(y_te, ens_te) if len(set(y_te)) > 1 else np.nan,
        'f1':          f1_score(y_te, yhat_ens, zero_division=0),
        'sensitivity': recall_score(y_te, yhat_ens, pos_label=1, zero_division=0),
        'specificity': recall_score(y_te, yhat_ens, pos_label=0, zero_division=0),
    })
    print(f'  [ENS] TEST    BA={fold_records[-1]["ba"]:.3f}  AUC={fold_records[-1]["auc"]:.3f}')

    # ── Stage 4 — SHAP per fold on outer-test predictions ───────────────────
    fold_shap = {}
    for m_name, est in fitted.items():
        try:
            # wrap the pipeline's predict_proba so SHAP sees the same features
            f = lambda data, est=est: est.predict_proba(data)[:, 1]
            # Use a subsample of training data as SHAP background
            bg_n = min(50, len(X_tr_full))
            bg_idx = np.random.RandomState(RANDOM_STATE + fold_i).choice(
                len(X_tr_full), size=bg_n, replace=False)
            explainer = shap.Explainer(f, X_tr_full.values[bg_idx],
                                        feature_names=feature_names)
            sv = explainer(X_te_full.values)
            fold_shap[m_name] = {
                'values':        sv.values,
                'data':          X_te_full.values,
                'feature_names': feature_names,
            }
        except Exception as err:
            print(f'    [SHAP {m_name}] skipped: {err}')
    per_fold_shap[fold_i] = fold_shap

    # Save SHAP + selected features artefacts
    for m_name, sh in fold_shap.items():
        np.savez(OUT_DIR / f'fold{fold_i}_shap_{m_name}.npz',
                 values=sh['values'], data=sh['data'],
                 feature_names=np.array(sh['feature_names']))
    per_fold_selections.append({
        'fold':             fold_i,
        'n_train':          len(tr_idx),
        'n_test':           len(te_idx),
        'top_k_cap':        top_k,
        'selected_features': ' | '.join(selected),
        'selected_scores':   json.dumps({k: round(v, 3) for k, v in sel_scores.items()}),
    })

print('\nAll outer folds completed.')

In [ ]:
# ── Aggregate per-fold results ───────────────────────────────────────────────

results_df = pd.DataFrame(fold_records)
results_df.to_csv(OUT_DIR / 'fold_results.csv', index=False)

summary = (
    results_df.groupby('model')[['ba', 'auc', 'f1', 'sensitivity', 'specificity']]
    .agg(['mean', 'std'])
    .round(3)
)
print('\n=== Per-model summary over outer folds ===')
print(summary.to_string())
summary.to_csv(OUT_DIR / 'model_summary.csv')

sel_df = pd.DataFrame(per_fold_selections)
sel_df.to_csv(OUT_DIR / 'per_fold_selected_features.csv', index=False)
print(f'\nArtefacts written to {OUT_DIR}/')
for p in sorted(OUT_DIR.iterdir()):
    print(' ', p.name)

In [ ]:
# ── Out-of-fold confusion matrix + ROC (ensemble) ────────────────────────────

mask = np.isfinite(oof_ensemble_prob)
y_oof = y[mask]
p_oof = oof_ensemble_prob[mask]

# Use the median per-fold threshold as the OOF decision boundary
med_thr = results_df.query('model == "ENS"')['threshold'].median()
yhat_oof = (p_oof >= med_thr).astype(int)

print(f'OOF ensemble : BA={balanced_accuracy_score(y_oof, yhat_oof):.3f} | '
      f'AUC={roc_auc_score(y_oof, p_oof):.3f} | thr={med_thr:.2f}')

cm = confusion_matrix(y_oof, yhat_oof)
fig, ax = plt.subplots(figsize=(4.5, 4))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(['improver', 'non-improver'])
ax.set_yticklabels(['improver', 'non-improver'])
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('OOF confusion (soft-vote ensemble)')
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black',
                fontsize=14)
plt.tight_layout()
fig.savefig(OUT_DIR / 'oof_confusion.png', dpi=200, bbox_inches='tight')
plt.show()

fpr, tpr, _ = roc_curve(y_oof, p_oof)
fig, ax = plt.subplots(figsize=(4.5, 4))
ax.plot(fpr, tpr, label=f'AUC = {roc_auc_score(y_oof, p_oof):.3f}', lw=2)
ax.plot([0, 1], [0, 1], '--', color='gray', lw=1)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('OOF ROC (soft-vote ensemble)')
ax.legend(loc='lower right')
plt.tight_layout()
fig.savefig(OUT_DIR / 'oof_roc.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ── Aggregate SHAP across folds (SVM head) ───────────────────────────────────
# Per-feature mean(|SHAP|) across folds — survives feature set differing per fold.

agg = defaultdict(list)
for fold_i, d in per_fold_shap.items():
    for m_name, sh in d.items():
        if m_name != 'SVM':
            continue
        vals = sh['values']
        for col_i, name in enumerate(sh['feature_names']):
            agg[name].extend(np.abs(vals[:, col_i]).tolist())

if agg:
    summary = (
        pd.DataFrame([
            {'feature': k,
             'mean_abs_shap': float(np.mean(v)),
             'n_test_obs':    len(v)}
            for k, v in agg.items()
        ])
        .sort_values('mean_abs_shap', ascending=False)
    )
    summary.head(25).to_csv(OUT_DIR / 'aggregated_shap_top25.csv', index=False)
    print(summary.head(20).to_string(index=False))

    top = summary.head(20).iloc[::-1]
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.barh(top['feature'], top['mean_abs_shap'], color='steelblue')
    ax.set_xlabel('mean |SHAP| across all OOF test points')
    ax.set_title('Top-20 features — SVM head, leakage-free CV')
    plt.tight_layout()
    fig.savefig(OUT_DIR / 'aggregated_shap.png', dpi=200, bbox_inches='tight')
    plt.show()
else:
    print('No SHAP values collected for SVM head — skipping aggregate plot.')

## Leakage audit
- `X_pool` is computed per-patient with no cross-patient normalisation — raw sheets only, no `Z_*`.
- `StandardScaler` / `SimpleImputer` sit inside every sklearn `Pipeline`, so they are fit only on each inner-train fold.
- Feature selection (`select_features_in_fold`) uses only `X_tr_pool` plus `StratifiedGroupKFold` seeded from the outer-train labels/groups.
- Hyperparameter search (`GridSearchCV`) consumes only outer-train data; it also uses `StratifiedGroupKFold` on patient groups.
- Threshold tuning is done on outer-train probabilities only, then applied unchanged to the outer-test fold.
- SHAP background is sampled from the outer-train matrix.
- `assert set(groups[tr]).isdisjoint(groups[te])` guards against patient-level leakage every fold.